# Pope Francis abroad: where he goes and what he says

Python for data science project, ENSAE, 2026. Authors: Quentin Brisemur, Arthur De Rouck, Adrien Vibert.

Between 2013 and 2024, Pope Francis made 47 trips outside Italy. In each country, he gave several speeches to very different audiences: heads of state, bishops, young people, refugees, leaders of other religions. This report asks two questions. Where does the pope go? And once there, does he adapt what he says to the country and to the audience?

The report has three parts.

1. Where does the pope go? We compare the countries visited by John Paul II, Benedict XVI and Francis with their religious composition and their level of development.
2. What does he say? We analyse the 342 speeches that Francis gave during his trips: who he speaks to, which themes he raises, and which words are specific to each continent.
3. Are the differences robust? We check the gaps found in part 2 with a bootstrap and a negative binomial regression.

The charts are interactive: hover over them to read the values.

The trips of the pope are both religious and diplomatic events. He meets heads of state, speaks before parliaments and international organisations, and his words are reported all over the world. Where he goes and what he says there tell us about the priorities of the Vatican. Before looking at the data, we expected Francis to favour poor and peripheral countries, in line with his call to go to "the peripheries". We also expected him to adapt his speeches strongly to each country, for example by talking about peace in countries at war and about migrants in countries of departure or arrival.

In [1]:
import sys
import textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "outils"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

import analyse_corpus as ac
import english as en
import graphiques as g
import robustness as rb
from stockage import fichier

g.style()
pd.set_option("display.max_colwidth", 70)

## Part 1. Where does the pope go?

This part places the trips of Francis among those of his two predecessors, John Paul II (1978-2005) and Benedict XVI (2005-2013). For each pope, we cross the list of visited countries with the religious composition of these countries and with their level of development. Three questions guide this part.

1. Who do the popes speak to? Are the visited countries mostly Catholic, or at least more Catholic than the others?
2. Growth or decline? Do the popes go where Catholicism is growing or where it is declining?
3. Development. Do they rather go to rich countries or to poor countries?

### The data

| Table | Source | Content | Years |
| --- | --- | --- | --- |
| Trips | vatican.va, scripts in `acquisition/` | one row per trip and visited country | 1979-2024 |
| Religions and population | Religious Characteristics of States Dataset, Demographics v. 2.0 (RCS-Dem 2.0) | population and share of each religion, Catholics included | every year up to 2015 |
| GDP per capita, region, income group | World Bank | GDP per capita in constant 2015 dollars | 1960-2024 |

No source lists the trips of the popes, so we scraped them from the Vatican website with `scraping_no_speeches.py`. The script follows the robots.txt file of the website and waits two seconds between two requests. `cleaning_no_speeches.py` then turns each trip title into the list of countries it names, with their ISO3 code. Trips inside Italy are left out. All the files are stored on the SSP Cloud and read directly by this notebook.

In [ ]:
POPES = ["John Paul II", "Benedict XVI", "Francis"]
PONTIFICATES = {"John Paul II": (1978, 2005), "Benedict XVI": (2005, 2013), "Francis": (2013, 2025)}
POPE_COLORS = {"Francis": g.COULEURS[0], "John Paul II": g.COULEURS[1], "Benedict XVI": g.COULEURS[2]}

FILE_TRIPS = "df_trips_countries.parquet"
FILE_RCS = "Religious Characteristics of States Dataset Project - Demographics v. 2.0 (RCS-Dem 2.0), COUNTRIES ONLY.SAV"
FILE_GDP = "API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
FILE_INCOME = "Metadata_Country_API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"

trips = pd.read_parquet(fichier(FILE_TRIPS))
trips = trips.rename(columns={"Pope": "pope", "Year": "year", "Place": "trip", "Country": "country", "ISO3": "iso3"})
trips["pope"] = trips["pope"].replace({"John-Paul II": "John Paul II"})
trips["year"] = trips["year"].astype(int)
# One trip = one trip title for a pope and a year
trips["trip_id"] = trips["pope"] + " " + trips["year"].astype(str) + " " + trips["trip"]
trips.head(3)

### Religions: RCS-Dem 2.0

The RCS-Dem 2.0 dataset, built at Princeton University, has one row per country and per year. The columns ending in `PC` are shares of the population (in %) and `TOTPP` is the total population. We keep the main religions and the detail of Christians. The part of the population that is in no category (Chinese folk religions, African religions, syncretic Christians...) goes into an `other_religions` column, except the people whose religion is unknown (`UNKPC`).

In [ ]:
rcs_raw = pd.read_spss(fichier(FILE_RCS))
# West Germany gets the ISO3 code of Germany
rcs_raw.loc[rcs_raw["ABBREV"] == "GFR", "ISO3"] = "DEU"

religions = pd.DataFrame({
    "iso3": rcs_raw["ISO3"].astype(str).str.strip(),
    "year": rcs_raw["YEAR"].astype(int),   # SPSS stores years as decimal numbers
    "population": rcs_raw["TOTPP"],
    "christians": rcs_raw["CHRPC"],
    "muslims": rcs_raw["MUSPC"],
    "jews": rcs_raw["JEWPC"],
    "buddhists": rcs_raw["BUDPC"],
    "hindus": rcs_raw["HINPC"],
    "no_religion": rcs_raw["NREPC"],
    "catholics": rcs_raw["CATPC"],
    "protestants": rcs_raw["PRTPC"],
    "orthodox": rcs_raw["ORTPC"],
    "unknown": rcs_raw["UNKPC"],
})

main_religions = ["christians", "muslims", "jews", "buddhists", "hindus", "no_religion"]
religions["other_religions"] = 100 - religions[main_religions].sum(axis=1) - religions["unknown"].fillna(0)
# Small negative values come from rounding in the data
religions["other_religions"] = religions["other_religions"].clip(lower=0)
# A country with no data on religions has no data on the other religions either
religions.loc[religions["christians"].isna(), "other_religions"] = np.nan
# Some former states have no ISO3 code
religions = religions[religions["iso3"] != ""]
religions.head(3)

### GDP per capita, region and income group (World Bank)

GDP per capita in constant 2015 dollars (indicator `NY.GDP.PCAP.KD`) comes from https://data.worldbank.org/indicator/NY.GDP.PCAP.KD. A second file gives the region and the income group of each country. The World Bank also gives the English name of each country, which we use in the charts.

In [ ]:
gdp_raw = pd.read_csv(fichier(FILE_GDP), skiprows=4)   # the first 4 lines are a title
gdp_raw = gdp_raw.dropna(axis=1, how="all")             # the lines end with a comma
names = gdp_raw.set_index("Country Code")["Country Name"]

# From a wide table (one column per year) to a long table (one row per country and year)
wb = gdp_raw.drop(columns=["Country Name", "Indicator Name", "Indicator Code"])
wb = wb.melt(id_vars="Country Code", var_name="year", value_name="gdp_per_capita")
wb = wb.rename(columns={"Country Code": "iso3"})
wb["year"] = wb["year"].astype(int)

wb_countries = pd.read_csv(fichier(FILE_INCOME))[["Country Code", "Region", "IncomeGroup"]]
wb_countries = wb_countries.rename(columns={"Country Code": "iso3", "Region": "region", "IncomeGroup": "income"})
# The aggregates (World, Euro area...) have no region
wb_countries = wb_countries.dropna(subset=["region"])
wb.head(3)

### Joining the trips with the closest year

RCS-Dem stops in 2015. For each visit, we take the closest observation with `merge_asof`, with at most 10 years of difference for religions and 5 years for GDP. For the visits after 2015 (Francis), we therefore use the 2015 data on religions.

In [ ]:
religions = religions.rename(columns={"year": "year_religion"}).sort_values("year_religion")
visits = pd.merge_asof(trips.sort_values("year"), religions, left_on="year", right_on="year_religion",
                       by="iso3", direction="nearest", tolerance=10)
visits["gap_religion"] = visits["year_religion"] - visits["year"]

wb_sorted = wb.rename(columns={"year": "year_wb"}).sort_values("year_wb")
visits = pd.merge_asof(visits.sort_values("year"), wb_sorted, left_on="year", right_on="year_wb",
                       by="iso3", direction="nearest", tolerance=5)
visits = visits.merge(wb_countries, on="iso3", how="left")

# Majority religion, among the main religions and the other religions
majority_columns = main_religions + ["other_religions"]
visits["majority"] = visits[majority_columns].dropna(how="all").idxmax(axis=1)

# Check: visits without data, and mean gap between the visit and the religious data
visits.groupby("pope").agg(
    visits=("iso3", "size"),
    no_religion_data=("christians", lambda x: x.isna().sum()),
    no_gdp_data=("gdp_per_capita", lambda x: x.isna().sum()),
    mean_gap_religion=("gap_religion", lambda x: x.abs().mean()),
).loc[POPES].round(1)

### How often do the popes travel?

In [ ]:
overview = trips.groupby("pope").agg(
    trips=("trip_id", "nunique"),
    visits=("iso3", "size"),
    countries=("iso3", "nunique"),
).loc[POPES]
overview["years_of_pontificate"] = [end - start for start, end in PONTIFICATES.values()]
overview["trips_per_year"] = (overview["trips"] / overview["years_of_pontificate"]).round(1)
overview

In [ ]:
per_year = trips.groupby(["year", "pope"])["trip_id"].nunique().rename("trips").reset_index()

fig = px.bar(per_year, x="year", y="trips", color="pope",
             category_orders={"pope": POPES}, color_discrete_map=POPE_COLORS)
fig.update_traces(marker_line_color=g.FOND, marker_line_width=1,
                  hovertemplate="%{x}, %{fullData.name}: %{y} trip(s)<extra></extra>")
fig.update_xaxes(title=None)
fig.update_yaxes(title="Trips", showgrid=True)
g.titre(fig, "Trips outside Italy each year", "Click on a pope in the legend to hide him", hauteur=430)
fig.show()

John Paul II and Francis have been more active than Benedict XVI, with about 4 trips per year on average for the first two against 3 for the latter. John Paul II took advantage of his youth, since he was elected at only 58. Francis, who was almost the same age as Benedict XVI when he was elected, shows a strong willingness to visit countries despite his age. He travelled more than John Paul II at the same age.

In [ ]:
per_country = (trips.groupby(["country", "pope"])["trip_id"].nunique()
                    .unstack(fill_value=0).reindex(columns=POPES, fill_value=0))
top = per_country.loc[per_country.sum(axis=1).sort_values().index[-15:]]

fig = go.Figure([go.Bar(x=top[pope], y=top.index, name=pope, orientation="h", marker_color=POPE_COLORS[pope],
                        marker_line_color=g.FOND, marker_line_width=1,
                        hovertemplate="%{y}, " + pope + ": %{x} trip(s)<extra></extra>")
                 for pope in POPES])
fig.update_layout(barmode="stack")
fig.update_xaxes(title="Trips", showgrid=True)
g.titre(fig, "The 15 most visited countries", "Number of trips that include the country", hauteur=520)
fig.show()

The most visited countries are either European or North American. The figure for Poland surprised us, but the peculiarity disappears once we see that John Paul II, born in Poland, accounts for most of these trips.

### Where do they go?

The buttons above the map show one pope at a time.

In [ ]:
visited = (trips.groupby(["iso3", "pope"]).size().unstack(fill_value=0)
                .reindex(columns=POPES, fill_value=0) > 0)
country = trips.groupby("iso3")["country"].first()
n_popes = visited.sum(axis=1)

fig = go.Figure()
fig.add_choropleth(locations=visited.index, z=n_popes, text=country[visited.index], zmin=0.5, zmax=3.5,
                   colorscale=[[0, g.BLEUS[2]], [1 / 3, g.BLEUS[2]], [1 / 3, g.BLEUS[6]],
                               [2 / 3, g.BLEUS[6]], [2 / 3, g.BLEUS[11]], [1, g.BLEUS[11]]],
                   colorbar=dict(title="Popes", tickvals=[1, 2, 3], thickness=12, len=0.5),
                   hovertemplate="%{text}: visited by %{z} of the three popes<extra></extra>")
for pope in POPES:
    places = visited.index[visited[pope]]
    fig.add_choropleth(locations=places, z=[1] * len(places), text=country[places], visible=False,
                       showscale=False, colorscale=[[0, POPE_COLORS[pope]], [1, POPE_COLORS[pope]]],
                       hovertemplate="%{text}<extra>" + pope + "</extra>")
fig.update_traces(marker_line_color="white", marker_line_width=0.5)

views = ["All three popes"] + POPES
buttons = [dict(label=view, method="update", args=[{"visible": [i == k for i in range(len(views))]}])
           for k, view in enumerate(views)]
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, showactive=True,
                                    x=0, xanchor="left", y=1.01, yanchor="bottom")],
                  margin=dict(l=0, r=0, b=0))
fig.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False, lataxis_range=[-58, 85],
                showcountries=True, countrycolor="white", landcolor="#ecebe6", bgcolor=g.FOND)
g.titre(fig, "Countries visited by the popes", "Hover over a country to see its name", hauteur=560, marge_haute=45)
fig.show()

The maps show large differences between popes. Together, they have visited most countries, except Russia and China (the largest countries never visited), the Horn of Africa and Central Asia. John Paul II roamed almost all of Europe and America, and most African countries. Benedict XVI focused on Europe and on key countries in America (United States, Mexico and Brazil). The comparison between Francis and John Paul II is interesting. Both visited many countries in America, but Francis picked mostly Eastern European and Asian countries, when his predecessor visited all of Europe and Africa.

The table below lists the countries that Francis was the first of the three popes to visit.

In [ ]:
before_francis = set(trips.loc[trips["pope"] != "Francis", "iso3"])
first_visits = trips[(trips["pope"] == "Francis") & ~trips["iso3"].isin(before_francis)]
first_visits.groupby("country")["year"].min().sort_values().rename("year of the visit").to_frame()

> To write. Where are these countries? What does this choice say about the priorities of Francis?

### Who do the popes speak to?

To compare the visited countries with the others, we build a table with one row per pope and per country of the world, and a column `visited` which is True if the pope went there. Each country counts once, even if the pope went there several times. We use the religious data of the middle year of each pontificate (2015 for Francis, because RCS-Dem stops in 2015).

In [ ]:
rows = []
for pope, (start, end) in PONTIFICATES.items():
    middle_year = min((start + end) // 2, 2015)
    table = religions[religions["year_religion"] == middle_year].copy()
    table["pope"] = pope
    table["visited"] = table["iso3"].isin(trips.loc[trips["pope"] == pope, "iso3"])
    rows.append(table)

countries = pd.concat(rows, ignore_index=True)
countries["majority"] = countries[majority_columns].dropna(how="all").idxmax(axis=1)
countries["name"] = countries["iso3"].map(names).fillna(countries["iso3"])
countries["status"] = np.where(countries["visited"], "Visited", "Not visited")

# Number of visited countries and of countries with data, for each pope
countries.groupby("pope")["visited"].agg(visited="sum", countries="size").loc[POPES]

In [ ]:
fig = px.box(countries.dropna(subset=["catholics"]), x="catholics", y="pope", color="status",
             points="all", hover_name="name", orientation="h",
             category_orders={"pope": POPES, "status": ["Visited", "Not visited"]},
             color_discrete_map={"Visited": g.COULEURS[0], "Not visited": g.GRIS})
fig.update_traces(marker=dict(size=5, opacity=0.5), line_width=1.5, jitter=0.6,
                  hovertemplate="%{hovertext}: %{x:.0f} % of Catholics<extra></extra>")
fig.update_layout(boxmode="group", boxgap=0.25)
fig.update_xaxes(title="Share of Catholics in the population (%)", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Share of Catholics in visited and other countries",
        "One point per country, the box covers the middle half of the countries", hauteur=520)
fig.show()

The table compares, for each pope, the visited countries with the others. A Christian country is a country where Christians are the largest religion. A Catholic country is a country where Catholics are more than half of the population. The world share of Catholics weights each country by its population.

In [ ]:
rows = []
for pope in POPES:
    table = countries[(countries["pope"] == pope) & countries["catholics"].notna()]
    visited_countries = table[table["visited"]]
    with_population = table.dropna(subset=["population"])
    rows.append({
        "pope": pope,
        "visited countries": len(visited_countries),
        "Christian countries (%)": (visited_countries["majority"] == "christians").mean() * 100,
        "Catholic countries (%)": (visited_countries["catholics"] > 50).mean() * 100,
        "Catholics, visited countries (%)": visited_countries["catholics"].mean(),
        "Catholics, other countries (%)": table.loc[~table["visited"], "catholics"].mean(),
        "Catholics, world (%)": np.average(with_population["catholics"], weights=with_population["population"]),
    })
pd.DataFrame(rows).set_index("pope").round(1)

In [ ]:
# Mean share of Catholics in the visited countries, by World Bank region (data of the year of the visit)
visits.groupby(["region", "pope"])["catholics"].mean().unstack()[POPES].round(1)

> To write. Are the visited countries more Catholic than the others? Is Francis different from his predecessors? Which regions explain the gap?

### Growth or decline?

For each visit, we compute the change in the share of Catholics in the visited country during the 5 years before the visit (in percentage points). We compare it with the change in the average share of Catholics of all countries during the same 5 years, each country counting for one. RCS-Dem stops in 2015, so for the visits after 2015 we use the change between 2010 and 2015.

In [ ]:
mean_catholics = religions.groupby("year_religion")["catholics"].mean().rename("mean_catholics").reset_index()

fig = px.line(mean_catholics[mean_catholics["year_religion"] >= 1974], x="year_religion", y="mean_catholics")
fig.update_traces(line=dict(color=g.COULEURS[0], width=2),
                  hovertemplate="%{x}: %{y:.1f} % on average<extra></extra>")
fig.update_xaxes(title=None)
fig.update_yaxes(title="Share of Catholics (%)", showgrid=True)
g.titre(fig, "Average share of Catholics across the countries of the world", "Each country counts for one", hauteur=400)
fig.show()

In [ ]:
visits["year_end"] = visits["year"].clip(upper=2015)
visits["year_start"] = visits["year_end"] - 5

# Share of Catholics in the visited country at the start and at the end of the 5 years
share = religions[["iso3", "year_religion", "catholics"]]
visits = visits.merge(share.rename(columns={"year_religion": "year_start", "catholics": "catholics_start"}),
                      on=["iso3", "year_start"], how="left")
visits = visits.merge(share.rename(columns={"year_religion": "year_end", "catholics": "catholics_end"}),
                      on=["iso3", "year_end"], how="left")

# Average share of all countries at the start and at the end
visits = visits.merge(mean_catholics.rename(columns={"year_religion": "year_start", "mean_catholics": "mean_start"}),
                      on="year_start", how="left")
visits = visits.merge(mean_catholics.rename(columns={"year_religion": "year_end", "mean_catholics": "mean_end"}),
                      on="year_end", how="left")

visits["change_country"] = visits["catholics_end"] - visits["catholics_start"]
visits["change_mean"] = visits["mean_end"] - visits["mean_start"]
visits["change_vs_mean"] = visits["change_country"] - visits["change_mean"]

changes = visits.dropna(subset=["change_country"]).copy()
summary = changes.groupby("pope")[["change_country", "change_mean", "change_vs_mean"]].mean()
summary["growing (%)"] = changes.groupby("pope")["change_country"].apply(lambda x: (x > 0).mean() * 100)
summary["faster than the mean (%)"] = changes.groupby("pope")["change_vs_mean"].apply(lambda x: (x > 0).mean() * 100)
summary.loc[POPES].round(2)

In [ ]:
fig = px.box(changes, x="change_vs_mean", y="pope", color="pope", points="all", hover_name="country",
             custom_data=["year"], orientation="h",
             category_orders={"pope": POPES}, color_discrete_map=POPE_COLORS)
fig.update_traces(marker=dict(size=5, opacity=0.5), line_width=1.5, jitter=0.6, showlegend=False,
                  hovertemplate="%{hovertext} %{customdata[0]}: %{x:+.1f} points<extra></extra>")
fig.add_vline(x=0, line=dict(color=g.ENCRE_2, width=1, dash="dot"))
fig.update_xaxes(title="Change compared with the average of all countries (percentage points)", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Change in the share of Catholics during the 5 years before the visit",
        "One point per visit. Right of the dotted line, Catholicism grew faster than on average.", hauteur=460)
fig.show()

> To write. Do the popes go where Catholicism is growing or where it is declining? Is there a difference between them?

### Development: do the popes go to rich or to poor countries?

For each visit, we divide the GDP per capita of the visited country by the world GDP per capita of the same year (World Bank code `WLD`). A ratio above 1 means that the visited country is richer than the world average. We use the median and not the mean, because a few very rich countries (Luxembourg, Switzerland...) pull the mean up.

In [ ]:
world_gdp = wb[wb["iso3"] == "WLD"][["year", "gdp_per_capita"]].rename(columns={"gdp_per_capita": "world_gdp_per_capita"})
visits = visits.merge(world_gdp, on="year", how="left")
visits["gdp_ratio"] = visits["gdp_per_capita"] / visits["world_gdp_per_capita"]

gdp = visits.dropna(subset=["gdp_ratio"])
gdp.groupby("pope").agg(
    visits=("gdp_ratio", "size"),
    median_gdp_visited=("gdp_per_capita", "median"),
    median_gdp_world=("world_gdp_per_capita", "median"),
    median_ratio=("gdp_ratio", "median"),
    richer_than_world=("gdp_ratio", lambda x: (x > 1).mean() * 100),
).loc[POPES].round(2)

The next chart uses the World Bank income groups. It is the current classification, not the one of the year of the visit.

In [ ]:
income_order = ["Low income", "Lower middle income", "Upper middle income", "High income"]
by_income = (visits.groupby(["income", "pope"]).size().rename("visits").reset_index())

fig = px.bar(by_income, x="income", y="visits", color="pope", barmode="group",
             category_orders={"income": income_order, "pope": POPES}, color_discrete_map=POPE_COLORS)
fig.update_traces(marker_line_color=g.FOND, marker_line_width=2,
                  hovertemplate="%{x}, %{fullData.name}: %{y} visits<extra></extra>")
fig.update_xaxes(title=None)
fig.update_yaxes(title="Visits", showgrid=True)
g.titre(fig, "Visits by income group of the visited country", hauteur=440)
fig.show()

In [ ]:
# Share of the countries of each income group that each pope visited (%)
countries_by_income = (visits.groupby(["income", "pope"])["iso3"].nunique()
                             .unstack(fill_value=0).loc[income_order, POPES])
countries_in_group = wb_countries["income"].value_counts()
(countries_by_income.div(countries_in_group, axis=0) * 100).round(1).assign(countries=countries_in_group)

> To write. Do the popes go rather to rich or to poor countries? Does Francis stand out, as his insistence on the poor would suggest?

## Part 2. What does he say?

We now look at what Francis says during his trips. We collected all the speeches he gave abroad and compare them by continent, by audience and over time.

### The speeches

The texts come from vatican.va. Two scripts of the `acquisition` folder produce the data.

1. `telecharger_discours.py` reads the list of trips outside Italy and their dates. It then goes through the monthly index of the speeches and keeps those given during a trip. This gives 415 texts.
2. `preparer_corpus.py` separates the header of each text (trip, audience, place, date) from the speech itself. It removes the press conferences on the plane, which mostly contain the questions of journalists, and two speeches given in the Vatican on the morning of a departure. 342 speeches remain.

The files are stored on the SSP Cloud. The notebook reads them directly, without running the collection again, which takes about ten minutes.

In [21]:
df = en.translate(ac.charger())
df[["date", "voyage", "pays", "public", "n_mots", "titre"]].head()

,date,voyage,pays,public,n_mots,titre
0,2013-07-22,Rio de Janeiro 2013,Brazil,Welcome and farewell,990,Welcome Ceremony at the Garden of Guanabara Palace (Rio de Janeiro...
1,2013-07-24,Rio de Janeiro 2013,Brazil,Young people and students,111,Greeting to the young people from Italy at the conclusion of his v...
2,2013-07-24,Rio de Janeiro 2013,Brazil,"The poor, the sick, migrants",999,Visit to the St. Francis of Assisi of the Providence of God Hospit...
3,2013-07-24,Rio de Janeiro 2013,Brazil,Prayer and memory,168,Impromptu address of Pope Francis from the balcony of the Basilica...
4,2013-07-25,Rio de Janeiro 2013,Brazil,Young people and students,2155,Welcoming Ceremony by the young people on the waterfront of Copaca...


The table has one row per speech. The function `ac.charger()` adds three pieces of information.

- The country and the continent. The place written in the header is compared with a list of countries and cities. If it is missing, we take the country of the trip when there is only one, otherwise the country of the other speeches of the same day.
- The audience, deduced from the words of the title ("Meeting with Authorities", "Meeting with Young People"...).
- The length of the speech in words.

The next table sums up the corpus and checks that every speech has a country.

In [22]:
pd.Series({
    "Speeches": len(df),
    "Trips": df["voyage"].nunique(),
    "Countries": df["pays"].nunique(),
    "Speeches without a country": df["pays"].isna().sum(),
    "Speeches of fewer than 150 words": (df["n_mots"] < 150).sum(),
    "Words in total": df["n_mots"].sum(),
}, name="Number").to_frame()

,Number
Speeches,342
Trips,47
Countries,66
Speeches without a country,0
Speeches of fewer than 150 words,8
Words in total,543217


### Where did Francis speak?

In [23]:
per_country = (df.groupby(["iso", "pays"])
                 .agg(speeches=("titre", "size"), trips=("voyage", "nunique"))
                 .reset_index())

fig = px.choropleth(per_country, locations="iso", color="speeches", hover_name="pays",
                    custom_data=["trips"], color_continuous_scale=g.BLEUS[2:])
fig.update_traces(marker_line_color="white", marker_line_width=0.5,
                  hovertemplate="%{hovertext}<br>%{z} speeches, %{customdata[0]} trip(s)<extra></extra>")
fig.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False, lataxis_range=[-58, 85],
                showcountries=True, countrycolor="white", landcolor="#ecebe6", bgcolor=g.FOND)
fig.update_layout(coloraxis_colorbar=dict(title="Speeches", thickness=12, len=0.6), margin=dict(l=0, r=0, b=0))
g.titre(fig, "Number of speeches given in each country", "Hover over a country to see the details", hauteur=500)
fig.show()

Francis spoke in 66 countries. Brazil (15 speeches), the United States (14) and Poland (13) come first, because each hosted one long trip with many meetings: the World Youth Days in Rio and Kraków, and the visit to Washington, New York and Philadelphia. A few countries were visited several times: France (3 trips), Cuba, Greece, Hungary and Portugal (2 trips each). Europe has the most countries (21) and the most speeches (105). Large parts of the world are missing: China, Russia, India, Australia and most of the Arabian Peninsula. Only 9 countries of sub-Saharan and North Africa were visited, Egypt being counted here with the Middle East.

### When?

In [24]:
per_year = df.groupby(["annee", "continent"]).size().rename("speeches").reset_index()
totals = df.groupby("annee").size().reindex(range(2013, 2025), fill_value=0)

fig = px.bar(per_year, x="annee", y="speeches", color="continent",
             category_orders={"continent": en.CONTINENT_ORDER}, color_discrete_map=en.CONTINENT_COLORS)
fig.update_traces(marker_line_color=g.FOND, marker_line_width=1.5,
                  hovertemplate="%{x}, %{fullData.name}: %{y} speeches<extra></extra>")
fig.add_scatter(x=totals.index, y=totals + 3, text=totals.where(totals > 0, ""), mode="text",
                showlegend=False, hoverinfo="skip", textfont=dict(color=g.ENCRE_2))
fig.add_annotation(x=2020, y=10, text="no trip", textangle=-90, showarrow=False, font=dict(size=12, color=g.GRIS))
fig.update_xaxes(title=None, dtick=1)
fig.update_yaxes(title="Speeches", showgrid=True)
g.titre(fig, "Speeches given each year, by continent",
        "Click on a continent in the legend to hide it", hauteur=500)
fig.show()

From 2014 to 2019, Francis made 4 to 7 trips a year. 2015 is the busiest year, with 64 speeches during 5 trips, including long journeys to South America, Cuba and the United States, and to Africa. There is no trip in 2020 because of the Covid-19 pandemic. After 2021, the pope gives fewer speeches each year (17 to 32, against 29 to 64 before), which may also reflect his age. The continents mix from one year to the next: there is no period dedicated to one region.

### Who does he speak to?

The audience is deduced from the title. For example "Meeting with the Authorities, Civil Society and the Diplomatic Corps" goes into "Authorities and diplomats".

In [25]:
audiences = df["public"].value_counts().sort_values()

fig = go.Figure(go.Bar(x=audiences.values, y=audiences.index, orientation="h", marker_color=g.COULEURS[0],
                       text=audiences.values, textposition="outside", cliponaxis=False,
                       hovertemplate="%{y}: %{x} speeches<extra></extra>"))
fig.update_xaxes(showgrid=True)
g.titre(fig, "Number of speeches by audience", hauteur=460)
fig.show()

Authorities come first: almost one speech in four (24 %) is given to political leaders, civil society or the diplomatic corps, and 41 of the 47 trips include such a meeting. The trips are therefore also state visits. Then come young people (16 %), ecumenical and interreligious meetings (15 %) and the poor, the sick and migrants (12 %). Bishops and clergy receive about one speech in five (19 %). Diplomacy and dialogue with other religions take as much room as the internal life of the Church.

### How long does he speak?

In [26]:
order = df.groupby("public")["n_mots"].median().sort_values().index.tolist()
medians = df.groupby("public")["n_mots"].median().reindex(order)

fig = px.strip(df, x="n_mots", y="public", hover_name="titre", custom_data=["voyage"],
               category_orders={"public": order[::-1]})
fig.update_traces(marker=dict(color=g.COULEURS[0], opacity=0.45, size=7), jitter=0.7,
                  hovertemplate="%{hovertext}<br>%{customdata[0]}, %{x} words<extra></extra>")
fig.add_scatter(x=medians.values, y=medians.index, mode="markers", showlegend=False,
                marker=dict(symbol="line-ns-open", size=26, color=g.ENCRE, line=dict(width=3, color=g.ENCRE)),
                hovertemplate="Median: %{x:.0f} words<extra></extra>")
fig.update_xaxes(title="Number of words", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Length of the speeches by audience",
        "One point per speech, the black line marks the median. Hover over a point to read its title.", hauteur=540)
fig.show()

The median speech has about 1,400 words. Speeches to bishops, clergy and families are the longest, with medians above 2,000 words: they discuss the life of the Church in detail. Prayers and memorials (median of 650 words) and welcome and farewell speeches (860 words) are the shortest. The longest speech, about 5,000 words, is the address to the World Meeting of Popular Movements in Bolivia in 2015, a programme on land, housing and work for the poor. The shortest ones are a few lines, such as the message written in the golden book of the State House in Nairobi (18 words).

### Does he talk about the country he is in?

We now measure how much Francis adapts his speeches, in three ways: the mention of the host country, the themes and the words specific to each continent. We end with the change over time. Oceania, with only 5 speeches, is left out of the comparisons by continent, as are the audiences with fewer than 10 speeches.

We first check whether the text names the country where the speech is given, for example "Kenya" in a speech in Nairobi.

In [27]:
df["names_host"] = ac.cite_le_pays_hote(df)

fig = make_subplots(rows=1, cols=2, subplot_titles=["By continent", "By audience"], horizontal_spacing=0.3)
for column, group in enumerate(["continent", "public"], start=1):
    shares = df.groupby(group)["names_host"].agg(["mean", "size"])
    shares = shares[shares["size"] >= 10].sort_values("mean")
    fig.add_bar(x=shares["mean"], y=shares.index, orientation="h", row=1, col=column, showlegend=False,
                width=0.5 if group == "continent" else 0.65,
                marker_color=[en.CONTINENT_COLORS.get(name, g.COULEURS[0]) for name in shares.index],
                text=[f"{p:.0%}" for p in shares["mean"]], textposition="outside", cliponaxis=False,
                customdata=shares["size"], hovertemplate="%{y}<br>%{x:.0%} of the %{customdata} speeches<extra></extra>")
fig.update_xaxes(range=[0, 1.1], tickformat=".0%", showgrid=True)
g.titre(fig, "Share of speeches that name the host country", "Groups of at least 10 speeches", hauteur=480)
fig.show()

Two speeches out of three (67 %) name the host country. The share is highest in Asia (85 %) and Africa (76 %), and lowest in Europe (56 %). By audience, the speeches to authorities almost always name the country (89 %), as do the welcome and farewell speeches (83 %), which address the whole nation. Speeches to young people (44 %) and the other meetings (30 %) name it least: they speak to a group rather than to a country. In Europe, part of the gap comes from speeches about Europe as a whole, as the specific words below show.

### What does he talk about on each continent?

We define 15 themes by lists of English words. For example, the theme "Peace and war" gathers "peace", "war", "weapons", "conflict"... For each continent, we count the mentions of each theme per 10,000 words. The measure is simple and easy to check, but it counts words and not ideas.

The colour reads row by row: the darkest cell shows the continent where the theme comes up most, the lightest the one where it comes up least. The lists of words are in `outils/analyse_corpus.py`.

In [28]:
large_continents = [c for c in en.CONTINENT_ORDER if (df["continent"] == c).sum() >= 10]
rates = ac.taux_lexique(df, "continent", en.LEXICON).loc[large_continents].T
rates = rates.loc[rates.mean(axis=1).sort_values(ascending=False).index]

g.carte_chaleur(rates, "Mentions of each theme per 10,000 words, by continent",
                "Each row goes from the lightest (minimum) to the darkest (maximum)",
                unite="mentions per 10,000 words").show()

Peace and war is the most frequent theme everywhere, but its weight varies a lot: 93 mentions per 10,000 words in the Middle East against 19 in the Americas. Interreligious dialogue follows the same pattern (31 against 2). In the Americas, the pope talks more about the family (29) and young people (28), in line with the World Youth Days of Rio and Panama and the World Meeting of Families in Philadelphia. Africa stands out for the economy and work (27), corruption and justice (11) and migrants (11). Europe stands out for mercy and forgiveness (16). Other themes stay stable, such as joy and hope (25 to 33) and memory and reconciliation (12 to 17). The gaps match the context of the countries visited: wars and religious diversity in the Middle East, poverty and corruption in Africa, a mostly Catholic population in Latin America.

### And by audience?

Same measure, comparing the audiences with at least 20 speeches.

In [29]:
large_audiences = df["public"].value_counts().loc[lambda n: n >= 20].index
rates = ac.taux_lexique(df, "public", en.LEXICON).loc[large_audiences].T
rates = rates.loc[rates.mean(axis=1).sort_values(ascending=False).index]
rates.columns = [textwrap.fill(a, 16).replace("\n", "<br>") for a in rates.columns]

g.carte_chaleur(rates, "Mentions of each theme per 10,000 words, by audience",
                "Each row goes from the lightest (minimum) to the darkest (maximum)",
                unite="mentions per 10,000 words").show()

The vocabulary changes even more with the audience than with the continent. Peace and war dominates the speeches to authorities (71), to other religions (64) and the prayers and memorials (81), but it is rare before bishops and clergy (15). Some themes are almost kept for one audience: young people in speeches to young people (59), interreligious dialogue in ecumenical meetings (33), migrants in meetings with the poor and migrants (18), the missionary Church before bishops (39) and clergy (26), mercy in prayers (46). Ecology is mostly addressed to authorities (18), and human dignity to authorities (14) and to the poor (15). The pope chooses his themes according to his listeners: public questions for leaders, the life of the Church for its clergy.

### The words specific to each continent

For each continent, we look for the words that are more frequent than in the rest of the corpus. The score, the log-odds with a Dirichlet prior (Monroe, Colaresi and Quinn, 2008), takes into account the size of each group and limits the weight of rare words. Above 2, the gap is clear. Place names logically come first.

In [30]:
words_by_continent = ac.mots_caracteristiques(df, "continent", n=10)
continents = [c for c in en.CONTINENT_ORDER if c in words_by_continent]

fig = make_subplots(rows=1, cols=len(continents), subplot_titles=continents, shared_xaxes=True,
                    horizontal_spacing=0.08)
for column, continent in enumerate(continents, start=1):
    scores = words_by_continent[continent].iloc[::-1]
    fig.add_bar(x=scores.values, y=scores.index, orientation="h", row=1, col=column, showlegend=False,
                marker_color=en.CONTINENT_COLORS[continent],
                hovertemplate="%{y}: score %{x:.1f}<extra></extra>")
highest = max(scores.max() for scores in words_by_continent.values())
fig.update_xaxes(range=[0, highest * 1.05], showgrid=True)
g.titre(fig, "The 10 most specific words of each continent",
        "z-score of the log-odds, above 2 the gap is clear", hauteur=480)
fig.show()

Beyond place names, each continent has its own words. In Europe, "roots", "bridge" and "union" come from the speeches on the European project, and "sea" and "island" from the Mediterranean trips (Lesvos, Malta, Cyprus, Marseille, Corsica). In the Americas, "indigenous" and "peoples" point to the meetings with indigenous peoples, and "family" and "youth" to the World Youth Days and the World Meeting of Families. In Asia, "harmony", "respect" and "religions" reflect countries where Catholics are a small minority, and "nuclear" the visit to Hiroshima and Nagasaki. In the Middle East, "Abraham", "fraternity" and "religions" refer to the dialogue with Islam and Judaism. In Africa, "violence" and "corruption" echo the conflicts in the countries visited.

### Is a speech closer to its country or to its audience?

To compare whole speeches rather than a few themes, we turn each speech into a vector of word weights (TF-IDF). A word weighs more when it is frequent in the speech and rare in the rest of the corpus. The similarity of two speeches is the cosine of the angle between their vectors: 0 when they share no word, 1 when they use the same words in the same proportions. We keep only the words written in lowercase. Place names, names of people and words such as "Kenyan" disappear, so two speeches in the same country do not look alike only because they name the same places.

We then compare the pairs of speeches: same country and different audience, same audience and different country, or neither.

In [31]:
pairs = ac.similarite_des_paires(df)
pairs["kind"] = np.select(
    [pairs["meme_pays"] & pairs["meme_public"], pairs["meme_pays"], pairs["meme_public"]],
    ["Same country, same audience", "Same country, other audience", "Same audience, other country"],
    "Other country, other audience")
pairs.groupby("kind")["similarite"].agg(mean_similarity="mean", pairs="size").sort_values("mean_similarity").round(3)

,mean_similarity,pairs
kind,,
"Other country, other audience",0.109,49087
"Same country, other audience",0.126,842
"Same audience, other country",0.147,8248
"Same country, same audience",0.151,134


In [32]:
# Each pair counts for both of its speeches
both = pd.concat([pairs, pairs.rename(columns={"i": "j", "j": "i"})])
both["audience"] = df["public"].to_numpy()[both["i"]]
COMPARED = {"Same audience, other country": g.COULEURS[0], "Same country, other audience": g.COULEURS[1]}

large_audiences = df["public"].value_counts().loc[lambda n: n >= 20].index
means = both[both["audience"].isin(large_audiences)].groupby(["audience", "kind"])["similarite"].mean().unstack()
means = means.sort_values("Same audience, other country")
baseline = pairs.loc[pairs["kind"] == "Other country, other audience", "similarite"].mean()

fig = go.Figure()
for audience, row in means.iterrows():
    fig.add_scatter(x=[row["Same country, other audience"], row["Same audience, other country"]], y=[audience] * 2,
                    mode="lines", line=dict(color=g.GRILLE, width=3), showlegend=False, hoverinfo="skip")
for kind, color in COMPARED.items():
    fig.add_scatter(x=means[kind], y=means.index, mode="markers", name=kind,
                    marker=dict(color=color, size=12, line=dict(color=g.FOND, width=1.5)),
                    hovertemplate="%{y}<br>" + kind + ": %{x:.3f}<extra></extra>")
fig.add_vline(x=baseline, line=dict(color=g.GRIS, width=1, dash="dot"),
              annotation_text="other country, other audience", annotation_position="bottom right",
              annotation_font=dict(size=12, color=g.GRIS))
fig.update_xaxes(title="Mean cosine similarity", showgrid=True)
g.titre(fig, "Is a speech closer to the same audience abroad or to its own country?",
        "Mean similarity of the speeches of each audience with two groups of other speeches", hauteur=480)
fig.show()

Two speeches given to the same audience in different countries are more alike (0.147 on average) than two speeches given in the same country to different audiences (0.126). Both are more alike than two speeches with nothing in common (0.109), so the place and the audience both count, but the audience counts more. 59 % of the speeches are closer to the speeches given to the same audience abroad than to the other speeches of their own country. The gap is largest for the institutional audiences: authorities, bishops and clergy receive speeches that look alike from one country to the next. On the contrary, speeches to the poor, the sick and migrants, and prayers and memorials, are closer to the other speeches of the same country. They are the most tied to the place.

### Does the host country shape the themes?

We now link the speeches to the data of part 1. For each host country, we take the share of Catholics and the GDP per capita at the time of the visit, and compare them with the rate of four themes in the speeches given there. We state four expectations before looking at the results. The pope should talk more about interreligious dialogue where Catholics are few, and more about the family where they are many. He should talk more about poverty, and about corruption and justice, in poorer countries. The rho is the Spearman rank correlation: -1 when the theme always decreases with the variable, 1 when it always increases, 0 when there is no link. We keep the countries with at least 3 speeches.

In [ ]:
# Religion and wealth of each host country, from part 1 (closest year to the visit)
host = visits[visits["pope"] == "Francis"].groupby("iso3")[["catholics", "gdp_per_capita"]].mean()
host = (df.groupby("iso").agg(country=("pays", "first"), speeches=("titre", "size"))
          .join(host).join(ac.taux_lexique(df, "iso", en.LEXICON)))
host = host[host["speeches"] >= 3].dropna(subset=["catholics", "gdp_per_capita"])

EXPECTATIONS = [("Interreligious dialogue", "catholics"), ("Family", "catholics"),
                ("Poverty and exclusion", "gdp_per_capita"), ("Corruption and justice", "gdp_per_capita")]
AXIS_TITLES = {"catholics": "Share of Catholics (%)", "gdp_per_capita": "GDP per capita (2015 dollars, log scale)"}

titles = []
for theme, variable in EXPECTATIONS:
    rho, p_value = spearmanr(host[variable], host[theme])
    titles.append(f"{theme} (rho = {rho:.2f}, p = {p_value:.2f})")

fig = make_subplots(rows=2, cols=2, subplot_titles=titles, vertical_spacing=0.2, horizontal_spacing=0.1)
for k, (theme, variable) in enumerate(EXPECTATIONS):
    row, column = k // 2 + 1, k % 2 + 1
    fig.add_scatter(x=host[variable], y=host[theme], mode="markers", row=row, col=column, showlegend=False,
                    text=host["country"], customdata=host["speeches"],
                    marker=dict(size=5 + host["speeches"], color=g.COULEURS[0], opacity=0.6,
                                line=dict(color=g.FOND, width=1)),
                    hovertemplate="%{text}, %{customdata} speeches<br>%{y:.1f} mentions per 10,000 words<extra></extra>")
    if variable == "gdp_per_capita":
        fig.update_xaxes(type="log", tickvals=[1000, 3000, 10000, 30000, 100000],
                         ticktext=["1k", "3k", "10k", "30k", "100k"], row=row, col=column)
    fig.update_xaxes(title_text=AXIS_TITLES[variable], row=row, col=column)
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True, rangemode="tozero", title_text="Mentions per 10,000 words", col=1)
fig.update_annotations(font_size=13)
g.titre(fig, "Four themes and the characteristics of the host country",
        f"One point per country with at least 3 speeches ({len(host)} countries), the size gives the number of speeches",
        hauteur=720)
fig.show()

> To write. Which expectations are confirmed? Comment on the strongest correlation and on a few countries that stand out.

### Does the discourse change over time?

We follow four themes year by year. The grey line gives the average over all trips. Some years have only one or two trips, so the changes there are fragile.

In [34]:
themes = ["Peace and war", "Migrants and refugees", "Ecology and creation", "Interreligious dialogue"]
by_year = ac.taux_lexique(df, "annee", en.LEXICON).reindex(range(2013, 2025))
average = ac.taux_lexique(df.assign(all_trips="all"), "all_trips", en.LEXICON).loc["all"]

fig = make_subplots(rows=2, cols=2, subplot_titles=themes, shared_xaxes=True,
                    vertical_spacing=0.16, horizontal_spacing=0.08)
for i, (theme, color) in enumerate(zip(themes, g.COULEURS)):
    row, column = i // 2 + 1, i % 2 + 1
    fig.add_scatter(x=by_year.index, y=by_year[theme], mode="lines+markers", row=row, col=column,
                    line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=g.FOND, width=1.5)),
                    showlegend=False, hovertemplate="%{x}: %{y:.1f} mentions per 10,000 words<extra></extra>")
    fig.add_hline(y=average[theme], line=dict(color=g.GRIS, width=1), layer="below", row=row, col=column)
fig.update_xaxes(dtick=2)
fig.update_yaxes(rangemode="tozero", showgrid=True)
g.titre(fig, "Mentions per 10,000 words, year by year",
        "The grey line gives the average over all trips", hauteur=580)
fig.show()

The peaks follow specific trips. Peace and war comes back strongly in 2014 (Holy Land, Strasbourg, Turkey), 2017 and 2022, the year of the war in Ukraine and of the trips to Kazakhstan and Bahrain. Migrants peak in 2016, with the visit to the refugee camp of Lesvos, and in 2019, with the trip to Morocco. Ecology peaks in 2015, the year of the encyclical Laudato si', and again in 2022. Interreligious dialogue rises in 2014, 2021 (Iraq) and 2022 (Kazakhstan, Bahrain). For these four themes there is no steady rise or fall over the period.

## Part 3. Are the differences robust?

This part follows the method proposed by Quentin. Part 2 compares rates of mentions between groups of speeches. Two doubts remain.

1. Chance. A continent has between 41 and 105 speeches, and one long speech centred on a single theme can be enough to move its rate. We answer this doubt with a bootstrap.
2. Confounding. The continent and the audience are linked. If peace comes up more in Africa, is it because of Africa, or because the pope meets more authorities there? We answer this doubt with a regression, which compares the continents for the same audience and the same year.

The code is in `outils/robustness.py`.

### Bootstrap: is a gap larger than chance?

The index of a theme on a continent is its rate of mentions on this continent divided by its rate in the whole corpus. An index of 2 means that the theme comes up twice as often as on average. To measure how fragile it is, we replay the trips 1,000 times. Each time, we draw at random, with replacement, as many speeches as the continent has (some come twice, others disappear) and compute the index again. The middle 95 % of the 1,000 values form the confidence interval.

- If the whole interval is above 1, the pope really talks more about the theme on this continent.
- If it is below 1, he really talks less about it.
- If it contains 1, we cannot conclude.

In [35]:
boot = rb.bootstrap(df, "continent", en.LEXICON, draws=1000, minimum=10)
continents = [c for c in en.CONTINENT_ORDER if c in set(boot["continent"])]
theme_order = list(ac.taux_lexique(df.assign(all_trips="all"), "all_trips", en.LEXICON).loc["all"].sort_values().index)

VERDICTS = {"more": ("More than average", g.COULEURS[0]),
            "less": ("Less than average", g.COULEURS[1]),
            "no clear gap": ("Interval contains 1", g.GRIS)}

fig = make_subplots(rows=1, cols=len(continents), subplot_titles=continents, shared_yaxes=True,
                    horizontal_spacing=0.015)
in_legend = set()
for column, continent in enumerate(continents, start=1):
    part = boot[boot["continent"] == continent].set_index("theme").loc[theme_order]
    for verdict, (label, color) in VERDICTS.items():
        sub = part[part["verdict"] == verdict]
        # One segment per theme, from the low end to the high end of the interval
        fig.add_scatter(x=np.column_stack([sub["low"], sub["high"], np.full(len(sub), np.nan)]).ravel(),
                        y=np.repeat(sub.index, 3), mode="lines", line=dict(color=color, width=2),
                        hoverinfo="skip", showlegend=False, row=1, col=column)
        fig.add_scatter(x=sub["index"], y=sub.index, mode="markers", name=label, legendgroup=verdict,
                        showlegend=verdict not in in_legend and len(sub) > 0, row=1, col=column,
                        marker=dict(color=color, size=9, line=dict(color=g.FOND, width=1.5)),
                        customdata=sub[["low", "high"]],
                        hovertemplate="%{y}<br>Index %{x:.2f}, interval from %{customdata[0]:.2f}"
                                      " to %{customdata[1]:.2f}<extra></extra>")
        if len(sub) > 0:
            in_legend.add(verdict)
fig.add_vline(x=1, line=dict(color=g.ENCRE_2, width=1, dash="dot"), row="all", col="all")
fig.update_xaxes(type="log", range=[np.log10(0.08), np.log10(6)], showgrid=True, tickangle=0,
                 tickvals=[0.25, 1, 4], ticktext=["1/4", "1", "4"])
fig.update_layout(legend_y=1.07)
g.titre(fig, "Index of each theme by continent, with its 95 % bootstrap interval",
        "1 is the average of all trips. Logarithmic scale: 2 and 1/2 are at the same distance from 1.",
        hauteur=660, marge_haute=30)
fig.show()

In [36]:
# Number of themes with a clear gap on each continent
pd.crosstab(boot["continent"], boot["verdict"]).loc[continents]

verdict,less,more,no clear gap
continent,,,
Europe,2,0,13
Americas,3,0,12
Asia,1,0,14
Middle East,5,2,8
Africa,1,1,13


Most gaps of part 2 do not survive the bootstrap: on each continent, the interval contains 1 for 8 to 14 themes out of 15. The Middle East is the most distinctive continent, with 7 clear gaps: more peace and more interreligious dialogue, and less young people, family, economy, missionary Church and poverty. In the Americas, the pope really talks less about peace, interreligious dialogue and migrants. In Africa, he talks more about the economy and work and less about the family. Europe and Asia stay close to the average. Some striking gaps of the heatmap rest on a few speeches only. Migrants in Africa have an index of 2, but its interval goes from 0.4 to 5.

### Regression: is it the place or the audience?

For each theme, we model the number of mentions in each speech. The explanatory variables are the continent, the audience and the year. The length of the speech enters as an offset, so the model explains a number of mentions per word, like the rates of part 2.

We use a negative binomial model rather than a Poisson model, because the counts are very dispersed: many speeches never mention a theme and a few mention it dozens of times. A Poisson model assumes that the variance of the counts equals their mean. The table shows that here it is between 3.6 and 19.5 times larger.

In [37]:
counts = rb.mentions(df, en.LEXICON)
pd.DataFrame({"mean": counts.mean(), "variance": counts.var(),
              "variance / mean": counts.var() / counts.mean()}).sort_values("variance / mean").round(1)

,mean,variance,variance / mean
Corruption and justice,1.1,3.8,3.6
Human dignity,1.1,4.5,4.0
Memory and reconciliation,2.1,9.4,4.5
Interreligious dialogue,1.6,7.9,4.8
The elderly,1.0,4.9,4.8
Joy and hope,4.8,26.8,5.5
Economy and work,3.1,17.3,5.6
Poverty and exclusion,2.1,11.8,5.7
Ecology and creation,1.3,7.6,5.9
Mercy and forgiveness,1.9,13.0,6.8


The reference continent is Europe and the reference audience is the authorities and diplomats, the largest group. Oceania (5 speeches) is left out, and the audiences with fewer than 10 speeches join the other meetings. The year enters as a linear trend.

Each coefficient is shown as a rate ratio. A ratio of 2 for Africa means twice as many mentions per word as in Europe, for the same audience and the same year. A ratio of 0.5 means half as many. With 15 themes and 13 variables, we run 195 tests, and some would come out significant by chance only. We therefore adjust the p-values with the Benjamini-Hochberg method and keep the effects with an adjusted p-value below 5 %.

In [38]:
data = df[df["continent"] != "Oceania"].copy()
small = data["public"].map(data["public"].value_counts()) < 10
data.loc[small, "public"] = "Other meetings"

reg = rb.regressions(data, en.LEXICON, reference_continent="Europe", reference_audience="Authorities and diplomats")
reg["significant"] = reg["p_adjusted"] < 0.05
reg.groupby("kind")["significant"].agg(significant="sum", tests="size")

,significant,tests
kind,,
Audience,56,120
Continent,15,60
Year,4,15


In [39]:
# The trend is shown over ten years rather than one, to make it readable
shown = reg.copy()
trend = shown["kind"] == "Year"
shown.loc[trend, ["rate_ratio", "low", "high"]] **= 10
shown.loc[trend, "variable"] = "Ten years later"

panels = {
    "Continent": [c for c in en.CONTINENT_ORDER if c in set(data["continent"]) - {"Europe"}],
    "Audience": [a for a in data["public"].value_counts().index if a != "Authorities and diplomats"],
    "Year": ["Ten years later"],
}
titles = ["Continent, vs Europe", "Audience, vs authorities", "Trend"]
rows = theme_order[::-1]   # the most frequent themes at the top
SHORT = {"Young people and students": "Young people", "Ecumenical and interreligious": "Ecumenical",
         "The poor, the sick, migrants": "Poor, sick, migrants", "Clergy and religious": "Clergy",
         "Prayer and memory": "Prayer, memory", "Other meetings": "Other", "Welcome and farewell": "Welcome, farewell"}

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=titles, horizontal_spacing=0.015,
                    column_widths=[len(columns) for columns in panels.values()])
for column, (kind, columns) in enumerate(panels.items(), start=1):
    table = shown[shown["kind"] == kind]
    pivot = {name: table.pivot(index="theme", columns="variable", values=name).loc[rows, columns]
             for name in ["rate_ratio", "low", "high", "p_adjusted", "significant"]}
    significant = pivot["significant"].astype(bool)
    # Colour only the significant effects, on a log scale capped at 4 and 1/4.
    # A ratio that cannot be estimated stays empty.
    ratio = pivot["rate_ratio"]
    color = np.log2(ratio).clip(-2, 2).where(significant | ratio.isna(), 0)
    text = ratio.map(lambda r: f"{r:.2g}").where(significant, "")
    fig.add_heatmap(z=color, x=[SHORT.get(c, c) for c in columns], y=rows,
                    text=text, texttemplate="%{text}", row=1, col=column,
                    customdata=np.dstack([pivot[name].to_numpy(dtype=float) for name in ["rate_ratio", "low", "high", "p_adjusted"]]),
                    zmin=-2, zmax=2, xgap=2, ygap=2, showscale=column == 3,
                    colorscale=[[0, "#c0461b"], [0.25, "#f4a47c"], [0.5, "#efeee9"], [0.75, g.BLEUS[4]], [1, g.BLEUS[10]]],
                    colorbar=dict(title="Rate ratio", tickvals=[-2, -1, 0, 1, 2], ticktext=["1/4", "1/2", "1", "2", "4"],
                                  thickness=12, len=0.6),
                    hovertemplate="%{y}, %{x}<br>Rate ratio %{customdata[0]:.2f}, 95 % interval from %{customdata[1]:.2f}"
                                  " to %{customdata[2]:.2f}<br>Adjusted p-value %{customdata[3]:.3f}<extra></extra>")
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(showline=False, tickangle=-45)
fig.update_layout(margin_b=120)
g.titre(fig, "Effect of the continent, the audience and the year on each theme",
        "Only the effects with an adjusted p-value below 5 % are coloured. Hover over a cell for the details.",
        hauteur=800)
fig.show()

One cell has no value: the migration theme never comes up in welcome and farewell speeches, so its ratio cannot be estimated.

Once the audience is taken into account, part of the continent effects remain. Compared with Europe, the pope talks about twice as much about peace in Africa (2.1) and in the Middle East (2.2), and about half as much in the Americas (0.57). Interreligious dialogue is four times rarer in the Americas (0.25) and more frequent in the Middle East (1.8). Corruption and justice come up three times more in Africa (2.9) and about twice more in Asia and the Americas. Migrants are eight times rarer in the Americas (0.12).

The audience matters more than the place: 56 audience effects out of 120 are significant, against 15 continent effects out of 60. Mercy is 2.5 to 20 times more frequent before any audience than before authorities. Public questions (peace, ecology, human dignity, corruption, economy) are up to eight times rarer before bishops and clergy.

The bootstrap and the regression agree on the main points: more peace and interreligious dialogue in the Middle East, less peace, interreligious dialogue and migrants in the Americas, more economy in Africa. Other gaps disappear: less family in Africa, or less economy and poverty in the Middle East, came from the type of meetings held there. Finally, four themes lose weight over time. In ten years, the family theme is divided by three (0.34), and poverty (0.53), human dignity (0.46) and the economy (0.63) by about two.

## Limits

- The speeches are the English versions published by the Vatican. Many of them were given in Spanish or Italian, so we often work on a translation.
- The country and the audience of each speech are deduced automatically from the header and the title. A few errors remain possible, for example "Georgia" can be the American state.
- The themes rely on lists of words that we chose. A word can have another meaning, and an idea can be expressed without these words.
- The groups have unequal sizes. Oceania has 5 speeches and some years only one trip. At the country level, most countries have fewer than 8 speeches, too few for a reliable bootstrap, which is why part 3 stays at the level of continents.
- RCS-Dem stops in 2015, so the religious composition of the countries visited by Francis after 2015 is the one of 2015. The World Bank income groups are the current ones, not those of the year of the visit.
- The regression shows associations, not causes. The programme of a trip, and so the audiences met, depends on the country visited.

- Speeches of the same trip share the context of the moment. In the bootstrap and the regression they are treated as independent, which makes the intervals somewhat too narrow.
- The similarity between speeches also depends on time: two speeches of the same trip are close in time, which can make them alike for reasons that have nothing to do with the country.

## Conclusion

> To write. What part 1 says about the countries Francis chose, compared with his predecessors.

Francis does adapt his speeches, but less to the country than to the audience. He names the host country in two speeches out of three, and some themes clearly follow the place: peace and interreligious dialogue in the Middle East, the economy and corruption in Africa, less peace and less dialogue in the Americas. These gaps resist the bootstrap and the regression. But the audience shapes the content much more. On average, a speech to bishops looks more like the speeches to bishops in other countries than like the other speeches of its own trip. Leaders hear about peace, ecology and human dignity, the clergy about mission and mercy, young people about youth and hope. The pope keeps a common message for each type of listener and adds to it the topics of the place he visits.